# GNN Metrics Evaluation
### Run in Colab with GPU runtime. Upload `merged_data.csv` and `gnn_pairs_discovery.pth` to the Colab sidebar before running.

In [ ]:
!pip install -q torch scikit-learn matplotlib seaborn networkx

## 1. Setup & Data Loading

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

df = pd.read_csv('merged_data.csv', index_col='Date', parse_dates=True)
threshold = len(df) * 0.7
df = df.dropna(thresh=threshold, axis=1)
df = df.ffill().bfill()
returns = df.pct_change().dropna()

scaler = StandardScaler()
scaled_returns = scaler.fit_transform(returns)
data_tensor = torch.tensor(scaled_returns.T, dtype=torch.float32)
stock_names = list(returns.columns)
num_stocks = len(stock_names)
lookback = 20

print(f'Stocks: {num_stocks} | Date range: {returns.index[0].date()} to {returns.index[-1].date()} | Trading days: {len(returns)}')

## 2. Rebuild Model & Load Weights

In [ ]:
class GraphDiscoveryNet(nn.Module):
    def __init__(self, num_nodes, lookback):
        super().__init__()
        self.adj_weights = nn.Parameter(torch.randn(num_nodes, num_nodes))
        self.gcn_layer   = nn.Linear(lookback, 64)
        self.output_layer = nn.Linear(64, 1)

    def forward(self, x):
        adj = torch.tanh(self.adj_weights)
        adj = adj + torch.eye(adj.size(0)).to(device)
        support = torch.relu(self.gcn_layer(x))
        output  = torch.matmul(adj, support)
        prediction = self.output_layer(output)
        return prediction, adj

model = GraphDiscoveryNet(num_stocks, lookback).to(device)
model.load_state_dict(torch.load('gnn_pairs_discovery.pth', map_location=device))
model.eval()
print('Model loaded successfully.')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total trainable parameters: {total_params:,}')

## 3. Generate Predictions & Core Metrics

In [ ]:
X_list, Y_list = [], []
for i in range(data_tensor.shape[1] - lookback):
    X_list.append(data_tensor[:, i:i+lookback])
    Y_list.append(data_tensor[:, i+lookback])

X_tensors = [t.to(device) for t in X_list]
Y_tensors = [t.to(device) for t in Y_list]

all_preds, all_actuals = [], []
with torch.no_grad():
    for i in range(len(X_tensors)):
        pred, _ = model(X_tensors[i])
        all_preds.append(pred.squeeze().cpu().numpy())
        all_actuals.append(Y_tensors[i].cpu().numpy())

all_preds   = np.array(all_preds)
all_actuals = np.array(all_actuals)

mse       = mean_squared_error(all_actuals, all_preds)
rmse      = np.sqrt(mse)
mae       = mean_absolute_error(all_actuals, all_preds)
hit_ratio = np.mean(np.sign(all_preds) == np.sign(all_actuals))

# Per-stock directional accuracy
per_stock_hit = np.mean(np.sign(all_preds) == np.sign(all_actuals), axis=0)

# Residuals
residuals = all_actuals - all_preds

# Baseline: predict zero (naive)
baseline_mse = mean_squared_error(all_actuals, np.zeros_like(all_preds))
r2 = 1 - mse / np.var(all_actuals)

print('=' * 40)
print('       GNN EVALUATION METRICS')
print('=' * 40)
print(f'MSE                  : {mse:.6f}')
print(f'RMSE                 : {rmse:.6f}')
print(f'MAE                  : {mae:.6f}')
print(f'R² Score             : {r2:.6f}')
print(f'Directional Hit Ratio: {hit_ratio:.4f}  ({hit_ratio*100:.2f}%)')
print(f'Baseline MSE (zero)  : {baseline_mse:.6f}')
print(f'MSE Reduction vs base: {((baseline_mse-mse)/baseline_mse)*100:.2f}%')
print('=' * 40)

## 4. Metric Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('GNN Model Evaluation — Core Metrics', fontsize=15, fontweight='bold', y=1.01)

# --- Plot 1: Predicted vs Actual (sample of stocks) ---
ax1 = axes[0, 0]
sample_idx = 0
ax1.scatter(all_actuals[:, sample_idx], all_preds[:, sample_idx],
            alpha=0.3, s=10, color='steelblue', label=stock_names[sample_idx])
lims = [min(all_actuals[:, sample_idx].min(), all_preds[:, sample_idx].min()),
        max(all_actuals[:, sample_idx].max(), all_preds[:, sample_idx].max())]
ax1.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
ax1.set_xlabel('Actual (scaled return)')
ax1.set_ylabel('Predicted (scaled return)')
ax1.set_title(f'Predicted vs Actual — {stock_names[sample_idx]}')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- Plot 2: Residual Distribution ---
ax2 = axes[0, 1]
ax2.hist(residuals.flatten(), bins=80, color='coral', edgecolor='white', alpha=0.85)
ax2.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax2.set_xlabel('Residual (Actual − Predicted)')
ax2.set_ylabel('Frequency')
ax2.set_title('Residual Distribution (all stocks & timesteps)')
ax2.grid(alpha=0.3)
mean_res = residuals.mean()
ax2.axvline(mean_res, color='red', linewidth=1, linestyle=':', label=f'Mean={mean_res:.4f}')
ax2.legend(fontsize=9)

# --- Plot 3: Per-stock Directional Hit Ratio ---
ax3 = axes[1, 0]
sorted_idx = np.argsort(per_stock_hit)[::-1]
colors = ['#2ecc71' if h >= 0.60 else '#e67e22' if h >= 0.55 else '#e74c3c' for h in per_stock_hit[sorted_idx]]
bars = ax3.bar(range(num_stocks), per_stock_hit[sorted_idx], color=colors)
ax3.set_xticks(range(num_stocks))
ax3.set_xticklabels([stock_names[i] for i in sorted_idx], rotation=90, fontsize=7)
ax3.axhline(0.5, color='black', linewidth=1, linestyle='--', label='Random baseline (50%)')
ax3.axhline(hit_ratio, color='steelblue', linewidth=1.2, linestyle='--', label=f'Overall avg ({hit_ratio*100:.1f}%)')
ax3.set_ylabel('Directional Hit Ratio')
ax3.set_title('Per-stock Directional Accuracy')
ax3.set_ylim(0.3, 0.85)
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)
patch_g = mpatches.Patch(color='#2ecc71', label='≥ 60%')
patch_o = mpatches.Patch(color='#e67e22', label='55–60%')
patch_r = mpatches.Patch(color='#e74c3c', label='< 55%')
ax3.legend(handles=[patch_g, patch_o, patch_r], fontsize=8, loc='lower right')

# --- Plot 4: Metric summary bar ---
ax4 = axes[1, 1]
metrics_names = ['MSE', 'RMSE', 'MAE', 'Hit Ratio', 'R²']
metrics_vals  = [mse, rmse, mae, hit_ratio, max(r2, 0)]
bar_colors = ['#3498db', '#9b59b6', '#1abc9c', '#2ecc71', '#e67e22']
ax4.barh(metrics_names, metrics_vals, color=bar_colors, height=0.5)
for i, v in enumerate(metrics_vals):
    ax4.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=10, fontweight='bold')
ax4.set_xlabel('Value')
ax4.set_title('Summary Metrics')
ax4.grid(axis='x', alpha=0.3)
ax4.set_xlim(0, max(metrics_vals) * 1.35)

plt.tight_layout()
plt.savefig('gnn_core_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gnn_core_metrics.png')

## 5. Learned Adjacency Matrix Heatmap

In [ ]:
with torch.no_grad():
    _, final_adj = model(X_tensors[-1])
    adj_matrix = final_adj.cpu().numpy()
    adj_no_diag = adj_matrix - np.eye(num_stocks)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Full adjacency matrix
sns.heatmap(adj_no_diag, xticklabels=stock_names, yticklabels=stock_names,
            cmap='RdYlGn', center=0, vmin=-0.8, vmax=0.8,
            ax=axes[0], linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Edge strength'})
axes[0].set_title('Learned Adjacency Matrix\n(diagonal removed)', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', labelsize=7, rotation=90)
axes[0].tick_params(axis='y', labelsize=7)

# Thresholded: strong connections only
threshold_val = 0.40
adj_thresh = np.where(np.abs(adj_no_diag) >= threshold_val, adj_no_diag, 0)
sns.heatmap(adj_thresh, xticklabels=stock_names, yticklabels=stock_names,
            cmap='RdYlGn', center=0, vmin=-0.8, vmax=0.8,
            ax=axes[1], linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Edge strength'})
axes[1].set_title(f'Strong Connections Only\n(|strength| ≥ {threshold_val})', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', labelsize=7, rotation=90)
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('gnn_adjacency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gnn_adjacency_heatmap.png')

## 6. GNN vs Pearson Correlation — Is the Model Learning Something New?

In [ ]:
pearson_corr = returns.corr().values  # [N x N]

# Flatten upper triangle (no diagonal)
triu_idx = np.triu_indices(num_stocks, k=1)
gnn_flat  = adj_no_diag[triu_idx]
corr_flat = pearson_corr[triu_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: GNN strength vs Pearson
ax = axes[0]
sc = ax.scatter(corr_flat, gnn_flat, alpha=0.35, s=12, c=np.abs(gnn_flat - corr_flat),
                cmap='plasma', edgecolors='none')
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.axvline(0, color='black', linewidth=0.7, linestyle='--')
lims = [-1, 1]
ax.plot(lims, lims, 'r--', linewidth=1, alpha=0.5, label='GNN = Pearson line')
plt.colorbar(sc, ax=ax, label='|GNN − Pearson|')
ax.set_xlabel('Pearson Correlation')
ax.set_ylabel('GNN Edge Strength')
ax.set_title('GNN Strength vs Pearson Correlation\n(each point = one stock pair)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

overall_corr = np.corrcoef(corr_flat, gnn_flat)[0, 1]
ax.text(0.05, 0.92, f'Correlation between measures: {overall_corr:.3f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# Distribution comparison
ax2 = axes[1]
ax2.hist(corr_flat, bins=50, alpha=0.6, color='steelblue', label='Pearson correlation', density=True)
ax2.hist(gnn_flat,  bins=50, alpha=0.6, color='coral',     label='GNN edge strength',  density=True)
ax2.set_xlabel('Value')
ax2.set_ylabel('Density')
ax2.set_title('Distribution: Pearson vs GNN Strengths')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gnn_vs_pearson.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlation between GNN and Pearson measures: {overall_corr:.4f}')
print('A low value here means the GNN is learning structure BEYOND pairwise correlation.')

## 7. Network Graph — Discovered Relationships

In [ ]:
SECTORS = {
    'LUPIN':'Pharma','DRREDDY':'Pharma','CIPLA':'Pharma','AUROPHARMA':'Pharma',
    'ALKEM':'Pharma','ZYDUSLIFE':'Pharma','SUNPHARMA':'Pharma','DIVISLAB':'Pharma',
    'BIOCON':'Pharma','TORNTPHARM':'Pharma',
    'WIPRO':'IT','MPHASIS':'IT','LTIM':'IT','TECHM':'IT','HCLTECH':'IT',
    'INFY':'IT','TCS':'IT','PERSISTENT':'IT','COFORGE':'IT','OFSS':'IT',
    'SBIN':'Banking','AXISBANK':'Banking','ICICIBANK':'Banking','HDFCBANK':'Banking',
    'BAJFINANCE':'Banking','ICICIPRULI':'Banking','SBILIFE':'Banking',
    'HDFCLIFE':'Banking','KOTAKBANK':'Banking','LICI':'Banking'
}
SECTOR_COLORS = {'IT':'#3498db', 'Pharma':'#2ecc71', 'Banking':'#e67e22', 'Other':'#95a5a6'}

EDGE_THRESH = 0.44  # match top-20 positive strength cutoff
G_pos = nx.DiGraph()
G_neg = nx.DiGraph()

for i in range(num_stocks):
    for j in range(num_stocks):
        if i == j: continue
        w = adj_no_diag[i, j]
        if w >= EDGE_THRESH:
            G_pos.add_edge(stock_names[i], stock_names[j], weight=w)
        elif w <= -EDGE_THRESH:
            G_neg.add_edge(stock_names[i], stock_names[j], weight=w)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

def draw_graph(G, ax, title, edge_color):
    if len(G.nodes) == 0:
        ax.set_title(title + '\n(no edges above threshold)')
        return
    pos = nx.spring_layout(G, seed=42, k=2.5)
    node_colors = [SECTOR_COLORS.get(SECTORS.get(n, 'Other'), '#95a5a6') for n in G.nodes]
    weights = [G[u][v]['weight'] for u, v in G.edges()]
    widths  = [abs(w) * 3 for w in weights]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=600, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color=edge_color, width=widths, alpha=0.6,
                           arrows=True, arrowsize=12, ax=ax,
                           connectionstyle='arc3,rad=0.1')
    legend_handles = [mpatches.Patch(color=c, label=s) for s, c in SECTOR_COLORS.items()]
    ax.legend(handles=legend_handles, loc='upper left', fontsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

draw_graph(G_pos, axes[0], f'Positive Edges  (strength ≥ {EDGE_THRESH})', '#27ae60')
draw_graph(G_neg, axes[1], f'Negative Edges  (strength ≤ −{EDGE_THRESH})', '#e74c3c')

plt.suptitle('GNN Discovered Stock Relationship Graph', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gnn_network_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gnn_network_graph.png')

## 8. Discovered Pairs — Ranked Table

In [ ]:
pearson_df = returns.corr()
pairs = []
for i in range(num_stocks):
    for j in range(num_stocks):
        if i == j: continue
        pairs.append({
            'Stock_A': stock_names[i],
            'Stock_B': stock_names[j],
            'GNN_Strength': adj_no_diag[i, j],
            'Pearson_Corr': pearson_df.loc[stock_names[i], stock_names[j]],
            'Sector_A': SECTORS.get(stock_names[i], 'Other'),
            'Sector_B': SECTORS.get(stock_names[j], 'Other')
        })

pairs_df = pd.DataFrame(pairs)
pairs_df['Same_Sector'] = pairs_df['Sector_A'] == pairs_df['Sector_B']
pairs_df['Abs_Strength'] = pairs_df['GNN_Strength'].abs()

top_pos = pairs_df[pairs_df['GNN_Strength'] > 0].nlargest(20, 'GNN_Strength')
top_neg = pairs_df[pairs_df['GNN_Strength'] < 0].nsmallest(20, 'GNN_Strength')

print('=== TOP 20 POSITIVE PAIRS ===')
print(top_pos[['Stock_A','Stock_B','GNN_Strength','Pearson_Corr','Sector_A','Sector_B','Same_Sector']].to_string(index=False))
print()
print('=== TOP 20 NEGATIVE PAIRS ===')
print(top_neg[['Stock_A','Stock_B','GNN_Strength','Pearson_Corr','Sector_A','Sector_B','Same_Sector']].to_string(index=False))

top_pos.to_csv('discovered_positive_pairs_eval.csv', index=False)
top_neg.to_csv('discovered_negative_pairs_eval.csv', index=False)
print('\nCSVs saved.')

## 9. Sparsity & Graph Statistics

In [ ]:
thresh_list = [0.2, 0.3, 0.4, 0.5, 0.6]
total_possible = num_stocks * (num_stocks - 1)

print('Edge sparsity at different thresholds:')
print(f'{"Threshold":>12} | {"Pos edges":>10} | {"Neg edges":>10} | {"Total":>8} | {"Density":>8}')
print('-' * 58)
for t in thresh_list:
    pos_e = int(np.sum(adj_no_diag >  t))
    neg_e = int(np.sum(adj_no_diag < -t))
    tot   = pos_e + neg_e
    dens  = tot / total_possible
    print(f'{t:>12.1f} | {pos_e:>10} | {neg_e:>10} | {tot:>8} | {dens:>7.2%}')

print(f'\nAdjacency matrix stats:')
print(f'  Mean abs weight : {np.abs(adj_no_diag).mean():.4f}')
print(f'  Std  abs weight : {np.abs(adj_no_diag).std():.4f}')
print(f'  Max  weight     : {adj_no_diag.max():.4f}')
print(f'  Min  weight     : {adj_no_diag.min():.4f}')

# Degree centrality
out_strength = np.abs(adj_no_diag).sum(axis=1)
top_hubs = np.argsort(out_strength)[::-1][:10]
print('\nTop hub stocks by total edge strength:')
for idx in top_hubs:
    print(f'  {stock_names[idx]:15s}: {out_strength[idx]:.4f}')

## 10. Save All Results Summary

In [ ]:
summary = {
    'Metric': ['MSE', 'RMSE', 'MAE', 'R2_Score', 'Directional_Hit_Ratio_%',
               'Baseline_MSE_zero', 'MSE_reduction_%', 'Num_Stocks', 'Trading_Days',
               'Total_Parameters', 'Lookback_Window'],
    'Value':  [round(mse,6), round(rmse,6), round(mae,6), round(r2,6),
               round(hit_ratio*100,2), round(baseline_mse,6),
               round(((baseline_mse-mse)/baseline_mse)*100,2),
               num_stocks, len(returns),
               sum(p.numel() for p in model.parameters()), lookback]
}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('gnn_evaluation_summary.csv', index=False)
print('All results saved:')
print('  gnn_core_metrics.png')
print('  gnn_adjacency_heatmap.png')
print('  gnn_vs_pearson.png')
print('  gnn_network_graph.png')
print('  discovered_positive_pairs_eval.csv')
print('  discovered_negative_pairs_eval.csv')
print('  gnn_evaluation_summary.csv')
print()
print(summary_df.to_string(index=False))